# BT 5: Revision analyses

This notebook consolidates the analyses added for the major revision (Discover Education). The canonical implementations live in `revision_pipeline/`; sections 2 to 4 invoke those scripts unchanged, while sections 5 and 6 are computed directly in the notebook from the committed data files.

Prerequisite: sections 2, 3 and 6 read the outputs of `retrain.py` (reproduction order in `revision_pipeline/README.md`). The setup cell points `NB_RESULTS` at the folder written by `retrain.py --outdir`.

Contents:

1. Setup
2. Supervised cross-validation (answers R1.4; encoder ablation answers R1.3)
3. SCS weight sensitivity (answers R1.7)
4. Expert agreement recomputed on the regenerated bank
5. Inter-rater reliability from the 8x30 expert vote counts
6. Threshold operating characteristics

## 1. Setup

Paths are repo relative, so the notebook must be started from the repository root. `revision_pipeline/` holds the canonical scripts and committed inputs; `NB_RESULTS` is the output folder of the `retrain.py` run (it contains `winner_and_constants.json`, `structural_scan_components.csv` and the regenerated `Verb List Classified by Model.csv`).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

LEVELS = ["Kn", "Cm", "Ap", "An", "Sn", "Ev"]
N_RATERS = 8

# Repo-relative inputs
NB_PIPE = Path("revision_pipeline")
NB_EXPERT_XLSX = Path("results") / "Final File(with model & Expert Classification).xlsx"
assert NB_PIPE.exists() and NB_EXPERT_XLSX.exists(), "Start this notebook from the repository root."

# Outputs of retrain.py (--outdir <folder> creates <folder>/results). Adjust if needed.
NB_RESULTS = Path("results")  # BT 3 writes the retrain outputs here
if not (NB_RESULTS / "winner_and_constants.json").exists():
    NB_RESULTS = NB_PIPE / "retrain_out" / "results"
if not NB_RESULTS.exists():
    alt = Path("..") / "pipeline_rerun" / "results"  # folder used for the paper run
    if alt.exists():
        NB_RESULTS = alt

# Forward-slash strings because these are passed on %run command lines below
NB_WINNER_JSON = (NB_RESULTS / "winner_and_constants.json").as_posix()
NB_COMPONENTS = (NB_RESULTS / "structural_scan_components.csv").as_posix()
NB_BANK = NB_RESULTS / "Verb List Classified by Model.csv"
print("retrain results folder:", NB_RESULTS, "| exists:", NB_RESULTS.exists())

retrain results folder: results | exists: True


## 2. Supervised cross-validation (answers R1.4)

`revision_pipeline/cv_metrics.py` runs repeated multi-label stratified 5-fold cross-validation (5 repeats, seeds 13 to 17) on the merged 361-verb core. Inside every fold it mirrors the full production decision rule: one-vs-rest fit with row weights, inner out-of-fold CDF calibration, and weighted per-level thresholds at the winner percentile taken from `winner_and_constants.json`. It evaluates the winner configuration, the model-zoo competitors, and two baselines, then writes `cv_results_<tag>.csv` and `cv_summary_<tag>.md` into `revision_pipeline/`.

Flags: `--encoder` and `--tag` select the sentence encoder, which gives the encoder ablation requested by R1.3. Defaults are the production encoder `sentence-transformers/all-mpnet-base-v2` with tag `mpnet`; the MiniLM and distilroberta runs are the commented lines in the next cell.

Note: this cell fits many models and can take up to an hour on CPU. The committed `cv_results_*.csv` and `cv_summary_*.md` files in `revision_pipeline/` are the archived outputs of exactly these commands, so the display cells below work without rerunning.

In [2]:
# Full CV under the production encoder; writes cv_results_mpnet.csv and cv_summary_mpnet.md
%run "revision_pipeline/cv_metrics.py" --winner-json $NB_WINNER_JSON

# Encoder ablation (R1.3), one run per encoder, same protocol:
# %run "revision_pipeline/cv_metrics.py" --winner-json $NB_WINNER_JSON --encoder sentence-transformers/all-MiniLM-L6-v2 --tag minilm
# %run "revision_pipeline/cv_metrics.py" --winner-json $NB_WINNER_JSON --encoder sentence-transformers/all-distilroberta-v1 --tag droberta

core: 381 verbs, 607 assignments


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  16%|█▌        | 32/199 [00:00<00:00, 250.87it/s]

Loading weights:  29%|██▉       | 58/199 [00:00<00:00, 235.88it/s]

Loading weights:  41%|████      | 82/199 [00:00<00:00, 229.79it/s]

...

CV: PassiveAggressive-C1


CV: kNN-21


CV: kNN-11


CV: NearestCentroid


# CV summary (mpnet, 381 core verbs, repeated 5x5 stratified CV, winner percentile P50)

| Model | macro AUC | macro F1 (mean over repeats) | 95% CI (seed 13 repeat) | micro F1 | subset acc |
|---|---|---|---|---|---|
| SGD-hinge-1e4 | 0.666 | 0.447 | [0.412, 0.481] | 0.450 | 0.093 |
| SGD-log-1e5 | 0.674 | 0.459 | [0.417, 0.484] | 0.460 | 0.097 |
| SGD-hinge-1e5 | 0.665 | 0.455 | [0.430, 0.495] | 0.460 | 0.097 |
| LinearSVC-C1 | 0.701 | 0.484 | [0.429, 0.501] | 0.485 | 0.128 |
| LogReg-C1 | 0.720 | 0.508 | [0.476, 0.548] | 0.508 | 0.156 |
| Ridge-a1 | 0.713 | 0.501 | [0.464, 0.533] | 0.500 | 0.135 |
| PassiveAggressive-C1 | 0.672 | 0.457 | [0.410, 0.475] | 0.458 | 0.098 |
| kNN-21 | 0.713 | 0.492 | [0.470, 0.537] | 0.496 | 0.142 |
saved: C:\Users\Muhammad Talha\Documents\Paper Review\Blooms-Taxonomy-Extension\revision_pipeline\cv_summary_mpnet.md


In [3]:
# Main CV summary for the production encoder
display(Markdown((NB_PIPE / "cv_summary_mpnet.md").read_text(encoding="utf-8")))

# CV summary (mpnet, 381 core verbs, repeated 5x5 stratified CV, winner percentile P50)

| Model | macro AUC | macro F1 (mean over repeats) | 95% CI (seed 13 repeat) | micro F1 | subset acc |
|---|---|---|---|---|---|
| SGD-hinge-1e4 | 0.666 | 0.447 | [0.412, 0.481] | 0.450 | 0.093 |
| SGD-log-1e5 | 0.674 | 0.459 | [0.417, 0.484] | 0.460 | 0.097 |
| SGD-hinge-1e5 | 0.665 | 0.455 | [0.430, 0.495] | 0.460 | 0.097 |
| LinearSVC-C1 | 0.701 | 0.484 | [0.429, 0.501] | 0.485 | 0.128 |
| LogReg-C1 | 0.720 | 0.508 | [0.476, 0.548] | 0.508 | 0.156 |
| Ridge-a1 | 0.713 | 0.501 | [0.464, 0.533] | 0.500 | 0.135 |
| PassiveAggressive-C1 | 0.672 | 0.457 | [0.410, 0.475] | 0.458 | 0.098 |
| kNN-21 | 0.713 | 0.492 | [0.470, 0.537] | 0.496 | 0.142 |
| kNN-11 | 0.706 | 0.473 | [0.454, 0.519] | 0.478 | 0.120 |
| NearestCentroid | 0.716 | 0.491 | [0.474, 0.541] | 0.494 | 0.148 |
| majority-level | nan | 0.090 | [0.081, 0.098] | 0.283 | 0.150 |
| freq-random | nan | 0.271 | [0.243, 0.311] | 0.297 | 0.043 |

Per-level F1 (winner SGD-hinge-1e4, mean over 5 repeats):
| Kn | Cm | Ap | An | Sn | Ev |
|---|---|---|---|---|---|
| 0.452 | 0.434 | 0.455 | 0.479 | 0.466 | 0.393 |

Paired exact McNemar (winner vs competitor, per verb-level decision, seed-13 repeat):
- vs SGD-log-1e5: winner-only correct 92, competitor-only correct 94, exact p = 0.9416
- vs SGD-hinge-1e5: winner-only correct 161, competitor-only correct 126, exact p = 0.04457
- vs LinearSVC-C1: winner-only correct 106, competitor-only correct 151, exact p = 0.00595
- vs LogReg-C1: winner-only correct 157, competitor-only correct 270, exact p = 5.013e-08
- vs Ridge-a1: winner-only correct 129, competitor-only correct 221, exact p = 1.003e-06
- vs PassiveAggressive-C1: winner-only correct 103, competitor-only correct 89, exact p = 0.3482
- vs kNN-21: winner-only correct 228, competitor-only correct 321, exact p = 8.315e-05
- vs kNN-11: winner-only correct 249, competitor-only correct 305, exact p = 0.01937
- vs NearestCentroid: winner-only correct 203, competitor-only correct 313, exact p = 1.466e-06
- vs majority-level: winner-only correct 397, competitor-only correct 401, exact p = 0.9154
- vs freq-random: winner-only correct 523, competitor-only correct 391, exact p = 1.424e-05

In [4]:
# Encoder ablation overview: mean over the 5 CV repeats, per model and encoder
tables = {}
for tag in ("mpnet", "minilm", "droberta"):
    f = NB_PIPE / f"cv_results_{tag}.csv"
    if f.exists():
        d = pd.read_csv(f)
        tables[tag] = d.groupby("model")[["macro_AUC", "macro_F1"]].mean().round(3)
ablation = pd.concat(tables, axis=1)
ablation.sort_values(("mpnet", "macro_F1"), ascending=False)

mpnet             minilm           droberta         
                     macro_AUC macro_F1 macro_AUC macro_F1 macro_AUC macro_F1
model                                                                        
LogReg-C1                0.720    0.508     0.698    0.489     0.683    0.469
Ridge-a1                 0.713    0.501     0.687    0.473     0.677    0.465
kNN-21                   0.713    0.492     0.703    0.481     0.660    0.438
NearestCentroid          0.716    0.491     0.702    0.490     0.681    0.472
LinearSVC-C1             0.701    0.484     0.674    0.460     0.669    0.455
kNN-11                   0.706    0.473     0.701    0.478     0.664    0.442
SGD-log-1e5              0.674    0.459     0.649    0.437     0.658    0.439
PassiveAggressive-C1     0.672    0.457     0.647    0.436     0.655    0.432
SGD-hinge-1e5            0.665    0.455     0.634    0.437     0.640    0.431
SGD-hinge-1e4            0.666    0.447     0.644    0.437     0.649    0.427
freq-random                NaN    0.271       NaN    0.271       NaN    0.271
majority-level             NaN    0.090       NaN    0.090       NaN    0.090

## 3. SCS sensitivity (answers R1.7)

`revision_pipeline/scs_sensitivity.py` re-aggregates the structural composite score SCS = wD x D - wM x M + wT x T + wC x C - wE x E from the per-candidate component statistics saved by `retrain.py`, re-applies the stored band penalty unchanged, and reports the winner's rank under single-weight perturbations (25 and 50 percent, both directions) and 200 joint random perturbations. No retraining is involved, so the cell runs in seconds. The report is written to `revision_pipeline/scs_sensitivity.md`.

In [5]:
%run "revision_pipeline/scs_sensitivity.py" --components $NB_COMPONENTS

# SCS sensitivity analysis

Formula check: max |recomputed - stored SCS| = 4.44e-16
Baseline weights {'wD': 1.0, 'wM': 1.0, 'wT': 0.8, 'wC': 0.3, 'wE': 0.05}; baseline winner: SGDClassifier
Baseline top 3: SGDClassifier (0.945), KNeighborsClassifier (0.908), PassiveAggressive (0.868)

| NearestCentroid | 0 | 1 | 4 |
| RidgeClassifier | 0 | 0 | 3 |
saved: C:\Users\Muhammad Talha\Documents\Paper Review\Blooms-Taxonomy-Extension\revision_pipeline\scs_sensitivity.md


In [6]:
display(Markdown((NB_PIPE / "scs_sensitivity.md").read_text(encoding="utf-8")))

# SCS sensitivity analysis

Formula check: max |recomputed - stored SCS| = 4.44e-16
Baseline weights {'wD': 1.0, 'wM': 1.0, 'wT': 0.8, 'wC': 0.3, 'wE': 0.05}; baseline winner: SGDClassifier
Baseline top 3: SGDClassifier (0.945), KNeighborsClassifier (0.908), PassiveAggressive (0.868)

| Perturbation | Winner | Baseline winner's rank |
|---|---|---|
| wD x0.5 | SGDClassifier | 1 |
| wD x0.75 | SGDClassifier | 1 |
| wD x1.25 | SGDClassifier | 1 |
| wD x1.5 | SGDClassifier | 1 |
| wM x0.5 | SGDClassifier | 1 |
| wM x0.75 | SGDClassifier | 1 |
| wM x1.25 | SGDClassifier | 1 |
| wM x1.5 | SGDClassifier | 1 |
| wT x0.5 | SGDClassifier | 1 |
| wT x0.75 | SGDClassifier | 1 |
| wT x1.25 | SGDClassifier | 1 |
| wT x1.5 | SGDClassifier | 1 |
| wC x0.5 | SGDClassifier | 1 |
| wC x0.75 | SGDClassifier | 1 |
| wC x1.25 | SGDClassifier | 1 |
| wC x1.5 | SGDClassifier | 1 |
| wE x0.5 | SGDClassifier | 1 |
| wE x0.75 | SGDClassifier | 1 |
| wE x1.25 | SGDClassifier | 1 |
| wE x1.5 | SGDClassifier | 1 |

Single-weight perturbations: winner rank 1 in 20/20, rank <=2 in 20/20
Joint random perturbations (200 draws, all weights x uniform[0.5,1.5]): rank 1 in 200/200, rank <=2 in 200/200

Who wins across the 200 joint draws (top-group stability):
| Model | rank 1 | in top 2 | in top 3 |
|---|---|---|---|
| SGDClassifier | 200 | 200 | 200 |
| PassiveAggressive | 0 | 176 | 197 |
| KNeighborsClassifier | 0 | 23 | 147 |
| LinearSVC | 0 | 0 | 49 |
| NearestCentroid | 0 | 1 | 4 |
| RidgeClassifier | 0 | 0 | 3 |

## 4. Expert agreement on the regenerated bank

`revision_pipeline/expert_recompute.py` recomputes every expert-agreement statistic against the regenerated bank. The 8x30 expert votes are unchanged, since votes attach to verbs rather than to model outputs. After normalization, "copying" resolves to the core verb "copy" and leaves the extension, so 29 of the 30 rated verbs remain matchable; verbs no longer hard-accepted are reported explicitly and the agreement metrics cover the hard-accepted matches. The script sets its input bank and report destination in its header block; adjust those constants if the retrain output lives in a different folder.

In [7]:
%run "revision_pipeline/expert_recompute.py"

# Expert agreement recomputed against the regenerated bank

Sampled verbs matched in the new bank: 29/30.
Not in the extension any more: ['copy'] ('copy' resolved to a core verb under normalization).
Matched but no longer hard-accepted by the new model: ['authorize', 'favour', 'rest', 'detach', 'lock', 'contain', 'renormalize']
Agreement metrics computed over the 22 hard-accepted matched verbs.

- Strict top-1: 11/22 = 50.0% (old model on 30 verbs: 20.0%)
- Endorsed top-1: 15/22 = 68.2% (old: 46.7%)
- Hard Jaccard/precision/recall: 0.28 / 0.67 / 0.32 (old: 0.17/0.46/0.18)
- Hard+soft: 0.38 / 0.69 / 0.45 (old: 0.31/0.53/0.42)
- Coverage of at least one endorsed level: hard 16/22, hard+soft 17/22 (old: 15/30, 23/30)

- McNemar hard vs hard+soft coverage: discordant 0 vs 1, exact p = 1.0000

- Consensus decomposition: moderate/high consensus 10/12, low consensus 5/10, Fisher exact p = 0.1718
- Moderate/high band vs conditional chance (53%): simulation p = 0.0243

- Score-vote correlation:

In [8]:
# %run leaves the script's top-level variables in the notebook namespace;
# lines holds the finished report.
display(Markdown("\n".join(lines)))

# Expert agreement recomputed against the regenerated bank

Sampled verbs matched in the new bank: 29/30.
Not in the extension any more: ['copy'] ('copy' resolved to a core verb under normalization).
Matched but no longer hard-accepted by the new model: ['authorize', 'favour', 'rest', 'detach', 'lock', 'contain', 'renormalize']
Agreement metrics computed over the 22 hard-accepted matched verbs.

- Strict top-1: 11/22 = 50.0% (old model on 30 verbs: 20.0%)
- Endorsed top-1: 15/22 = 68.2% (old: 46.7%)
- Hard Jaccard/precision/recall: 0.28 / 0.67 / 0.32 (old: 0.17/0.46/0.18)
- Hard+soft: 0.38 / 0.69 / 0.45 (old: 0.31/0.53/0.42)
- Coverage of at least one endorsed level: hard 16/22, hard+soft 17/22 (old: 15/30, 23/30)

- McNemar hard vs hard+soft coverage: discordant 0 vs 1, exact p = 1.0000

- Consensus decomposition: moderate/high consensus 10/12, low consensus 5/10, Fisher exact p = 0.1718
- Moderate/high band vs conditional chance (53%): simulation p = 0.0243

- Score-vote correlation: mean rho = 0.224, permutation p = 0.0087 (old model: 0.144, p = 0.072)

## 5. Inter-rater reliability from the expert vote counts

Eight experts rated each of the 30 sampled verbs and could endorse any subset of the six levels, or None. The committed expert file stores only per-verb vote counts (`cnt_Kn` to `cnt_Ev` and `cnt_None`, each 0 to 8), so each level is treated as its own binary rating task: every rater either endorsed the level for the verb or did not. Fleiss kappa for a binary category follows directly from those counts via observed versus expected pairwise agreement.

Kappa near zero at moderate prevalence means the experts' selections for that level are close to independent, in other words low consensus, consistent with the p_max analysis in the paper.

In [9]:
votes = pd.read_excel(NB_EXPERT_XLSX)
CATS = LEVELS + ["None"]
CNT = votes[[f"cnt_{c}" for c in CATS]].to_numpy(int)  # positive votes per verb, 0 to 8


def fleiss_kappa_binary(k, n):
    """Fleiss kappa for one binary category from per-item positive-vote counts k out of n raters."""
    k = np.asarray(k, float)
    p_obs = ((k * (k - 1) + (n - k) * (n - k - 1)) / (n * (n - 1))).mean()  # pairwise agreement
    prev = k.sum() / (k.size * n)
    p_exp = prev ** 2 + (1 - prev) ** 2
    kappa = np.nan if p_exp == 1.0 else (p_obs - p_exp) / (1 - p_exp)
    return kappa, p_obs, prev


rows = []
for j, cat in enumerate(CATS):
    kappa, p_obs, prev = fleiss_kappa_binary(CNT[:, j], N_RATERS)
    rows.append({"level": cat, "prevalence": round(prev, 2),
                 "observed agreement": round(p_obs, 2), "Fleiss kappa": round(kappa, 3)})
pd.DataFrame(rows)

,level,prevalence,observed agreement,Fleiss kappa
0,Kn,0.15,0.76,0.092
1,Cm,0.19,0.68,-0.022
2,Ap,0.29,0.60,0.020
3,An,0.18,0.73,0.081
4,Sn,0.21,0.67,0.011
5,Ev,0.19,0.73,0.132
6,None,0.28,0.61,0.039


Panel-size curve: how reliability would look with fewer raters. Raters are exchangeable, so for a panel of j raters the positive-vote count of a verb is a hypergeometric draw of j from the observed 8 votes. Each panel size uses 3000 simulated panels; the statistic is the per-level kappa averaged over the six taxonomy levels. The draw at j = 8 is deterministic and reproduces the table above. A curve that is flat by 6 to 8 raters indicates that additional raters would refine the estimates only marginally.

In [10]:
K6 = CNT[:, :6]  # the six taxonomy levels, None excluded
rng = np.random.default_rng(13)


def panel_kappas(j, iters=3000):
    vals = np.empty(iters)
    for i in range(iters):
        sub = rng.hypergeometric(K6, N_RATERS - K6, j)  # j-rater subsample per verb and level
        vals[i] = np.nanmean([fleiss_kappa_binary(sub[:, l], j)[0] for l in range(6)])
    return vals


curve = []
for j in range(2, N_RATERS + 1):
    v = panel_kappas(j)
    curve.append({"panel size": j, "mean kappa (6 levels)": round(v.mean(), 3),
                  "pct 2.5": round(np.percentile(v, 2.5), 3),
                  "pct 97.5": round(np.percentile(v, 97.5), 3)})
pd.DataFrame(curve)

,panel size,mean kappa (6 levels),pct 2.5,pct 97.5
0,2,0.041,-0.103,0.202
1,3,0.045,-0.038,0.138
2,4,0.048,-0.010,0.110
3,5,0.050,0.009,0.094
4,6,0.051,0.021,0.082
5,7,0.052,0.033,0.071
6,8,0.052,0.052,0.052


## 6. Threshold operating characteristics

User-facing view of the acceptance thresholds: the per-level hard thresholds are scaled by a factor from 1.0 down to 0.7, and a level counts as an entry when `cal_score >= scale x thr`. Entries are counted before the top-2 margin collapse, so the scale 1.0 row shows the raw threshold-pass density rather than the production bank's post-collapse 1.15 levels per verb. Scale 0.85 coincides with the pipeline's soft tier.

Columns: acceptance is the share of all candidates with at least one entry; expert precision and recall compare the predicted level set of each rated verb against its endorsed set (levels with at least 2 of 8 expert votes), averaged over the rated verbs that have at least one predicted level at that scale.

Caution: this table describes operating characteristics for downstream users. It must not be used to re-tune the production threshold, because the expert sample is the validation set and is far too small to tune on without contaminating the validation.

In [11]:
bank = pd.read_csv(NB_BANK)
bank["verb"] = bank["Verb"].astype(str).str.strip().str.lower()
cal = bank[[f"cal_score_{l}" for l in LEVELS]].to_numpy(float)
thr = bank[[f"thr_{l}" for l in LEVELS]].to_numpy(float)

# Rated verbs still present in the regenerated bank, with their endorsed level sets
votes["verb_clean"] = votes["Verb_clean"].astype(str).str.strip().str.lower()
rated = votes.merge(bank[["verb"]].assign(bank_row=np.arange(len(bank))),
                    left_on="verb_clean", right_on="verb", how="inner")
bank_rows = rated["bank_row"].to_numpy()
endorsed = rated[[f"cnt_{l}" for l in LEVELS]].to_numpy(int) >= 2  # at least 2 of 8 votes
print(f"{len(bank)} candidates in the bank; {len(rated)} of the 30 rated verbs matched")

1650 candidates in the bank; 29 of the 30 rated verbs matched


In [12]:
def sweep(scale):
    entries = cal >= thr * scale  # per-level entries before the top-2 collapse
    accepted = entries.any(axis=1)
    pred = entries[bank_rows]  # predicted level sets for the rated verbs
    inter = (pred & endorsed).sum(axis=1)
    has_pred = pred.sum(axis=1) > 0
    prec = (inter[has_pred] / pred.sum(axis=1)[has_pred]).mean()
    rec = (inter[has_pred] / endorsed.sum(axis=1)[has_pred]).mean()
    return {"scale": scale,
            "acceptance": round(accepted.mean(), 3),
            "levels per accepted verb": round(entries[accepted].sum() / accepted.sum(), 2),
            "expert precision": round(prec, 2),
            "expert recall": round(rec, 2)}


pd.DataFrame([sweep(s) for s in (1.00, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70)])

,scale,acceptance,levels per accepted verb,expert precision,expert recall
0,1.00,0.861,1.50,0.69,0.42
1,0.95,0.909,1.66,0.69,0.44
2,0.90,0.950,1.78,0.70,0.45
3,0.85,0.969,1.97,0.65,0.44
4,0.80,0.983,2.14,0.64,0.46
5,0.75,0.990,2.34,0.65,0.55
6,0.70,0.995,2.54,0.64,0.56


## 7. Evaluation upgrades: gate scoring, matched-acceptance comparison, independent encoders

Answers Reviewer 3's request to separate expert-side from model-side uncertainty (the
accept/reject gate is scored as its own decision against the experts' None votes), and
addresses the circularity concern about structural metrics by recomputing the ordinal
separation trend in encoders that played no part in training or selection. Also compares
model families at matched acceptance, which shows the structural objective and
cross-validated generalization are statistically independent (the basis of the joint
selection rule in the paper).

In [13]:
%run "revision_pipeline/evaluation_upgrade.py"

# Evaluation upgrades

## 1. Rejection-aware expert evaluation

Rated verbs present in the bank: 29 (accepted 26, rejected 3).

A rejection is a decision the experts can also judge, because they could vote
'None of the above'. Scoring only accepted verbs discards that evidence.

| | experts: None is plurality | experts: a level leads |
|---|---|---|
| model accepts | 4 | 22 |
| model rejects | 1 | 2 |

- Correct acceptances (model assigns, experts back a level): 22
- Correct rejections   (model declines, experts say None):   1
- False acceptances    (model assigns, experts say None):    4
- False rejections     (model declines, experts back a level): 2
- **Gate accuracy (accept/reject decision): 23/29 = 79.3%**
- Fisher exact on the 2x2: p = 0.4461

None-vote rates by model decision:
- accepted verbs: mean 2.19/8
- rejected verbs: mean 2.67/8
- Mann-Whitney (rejected have more None votes): p = 0.3841

Rejected verbs in detail:
  input          None 2/8, top level vote 4/8, expert-endor

The structural objective penalizes deviation from a target acceptance band, but
acceptance is set by the threshold percentile, not by model quality. Comparing
families at a COMMON acceptance removes that confound.

### At acceptance ~0.80

| family | percentile | acceptance | SCS | CV AUC |
|---|---|---|---|---|
| SGDClassifier | P55 | 0.804 | 0.998 | 0.666 |
| PassiveAggressive | P55 | 0.787 | 0.928 | 0.672 |
| LogisticRegression | P45 | 0.764 | 0.894 | 0.720 |
| KNeighborsClassifier | P55 | 0.779 | 0.876 | 0.713 |
| NearestCentroid | P40 | 0.825 | 0.863 | 0.716 |
| LinearSVC | P50 | 0.798 | 0.849 | 0.701 |
| RidgeClassifier | P45 | 0.802 | 0.712 | 0.713 |

Spearman(SCS at matched acceptance, CV AUC) = -0.414, p = 0.355

### At acceptance ~0.86

| family | percentile | acceptance | SCS | CV AUC |
|---|---|---|---|---|
| SGDClassifier | P50 | 0.870 | 0.933 | 0.666 |
| KNeighborsClassifier | P50 | 0.861 | 0.909 | 0.713 |
| LinearSVC | P45 | 0.856 | 0.899 | 0.701 |
| PassiveAggressive | 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4955.25it/s]

- mpnet (used in training): gap means [np.float64(0.1149), np.float64(0.0994), np.float64(0.1033), np.float64(0.1577), np.float64(0.1252)], Spearman rho = 0.60, exact one-tailed p = 0.1750


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:  43%|████▎     | 44/103 [00:00<00:00, 427.20it/s]

Loading weights:  84%|████████▍ | 87/103 [00:00<00:00, 392.57it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 466.11it/s]

- distilroberta (independent): gap means [np.float64(0.0612), np.float64(0.0532), np.float64(0.0513), np.float64(0.0865), np.float64(0.0672)], Spearman rho = 0.50, exact one-tailed p = 0.2250

If the monotone trend survives in encoders that were never used for training or
model selection, the ordinal structure of the extended bank is a property of the
assignments rather than an artifact of the embedding space used to produce them.

saved: C:\Users\Muhammad Talha\Documents\Paper Review\Blooms-Taxonomy-Extension\revision_pipeline\evaluation_upgrade.md


## 8. Held-out full-pipeline evaluation with ordinal metrics

The strongest part of the answer to Reviewer 1 comment 4 and the editor's evaluation
request: held-out core verbs pass through the COMPLETE production pipeline (calibration,
thresholds, top-2 collapse) as unseen candidates. Because every held-out verb is a known
Bloom verb, this provides the first labeled test of the acceptance gate (recall) and
ordinal error metrics that respect the taxonomy's level ordering. The score-averaging
ensemble is also tested here (and rejected on the evidence).

In [14]:
%run "revision_pipeline/methodology_upgrades.py"

# Methodology upgrade tests

## A + B. Held-out full-pipeline evaluation with ordinal metrics

Held-out core verbs pushed through the COMPLETE production pipeline (calibration,
P50 thresholds, top-2 collapse). All are known Bloom verbs, so the gate should
accept them; 5-fold multi-label stratified, means over folds:

| Pipeline | Gate recall | Primary-in-labels | Within-1-level | Mean ordinal dist | macro F1 | macro AUC |
|---|---|---|---|---|---|---|
| Ensemble(3, vote>=2) | 0.890 | 0.618 | 0.773 | 0.73 | 0.479 | 0.729 |
| LogReg-C1 | 0.895 | 0.593 | 0.736 | 0.81 | 0.489 | 0.720 |
| NearestCentroid | 0.882 | 0.597 | 0.736 | 0.81 | 0.479 | 0.722 |
| kNN-21 | 0.924 | 0.600 | 0.784 | 0.74 | 0.477 | 0.719 |

Reading guide: 'Gate recall' is the share of held-out KNOWN Bloom verbs that survive
the acceptance gate; this is the first labeled test of the gate. 'Primary-in-labels'
scores the primary domain against the verb's published level set. 'Within-1-level'
credits adjacent-level predictio

## 9. Paper-ready tables and the selection-invariance analysis

Regenerates every table reported in the revised manuscript from the final artifacts,
including the joint-selection comparison and its aggregation-invariance analysis (the
same production model is selected under maximin, rank-sum, weighted combinations, and
tolerance rules in either direction), which is the formal defense of the selection
criterion added in response to Reviewer 1 comment 7.

In [15]:
%run "revision_pipeline/paper_results.py"

# Paper-Ready Results (final pipeline, corrected core)

## Table M1. Model comparison and joint selection

| Family (best config) | Structural objective | Acceptance | CV macro AUC | CV macro F1 | Maximin |
|---|---|---|---|---|---|
| **KNeighborsClassifier (P50)** | 0.908 | 0.861 | 0.713 | 0.492 | **0.554** |
| LinearSVC (P45) | 0.898 | 0.856 | 0.701 | 0.484 | 0.496 |
| NearestCentroid (P40) | 0.860 | 0.825 | 0.716 | 0.491 | 0.268 |
| LogisticRegression (P45) | 0.857 | 0.764 | 0.720 | 0.508 | 0.252 |
| PassiveAggressive (P50) | 0.920 | 0.833 | 0.672 | 0.457 | 0.112 |
| RidgeClassifier (P40) | 0.815 | 0.857 | 0.713 | 0.501 | 0.000 |
| SGDClassifier (P55) | 0.983 | 0.804 | 0.666 | 0.447 | 0.000 |

Criteria independence: Spearman(structural objective, CV AUC) = -0.71 (p = 0.07); at matched acceptance the association is also non-significant, so the two criteria are treated as complementary and the winner maximizes the weaker normalized score (maximin).

Selection robustness: perturbing ev